In [17]:
import requests
import json
import numpy as np

In [18]:
PREDICT_URL = "http://127.0.0.1:8000/predict/"
RESULT_URL = "http://127.0.0.1:8000/result/"
RETRAIN_URL = "http://127.0.0.1:8000/retrain/"
NEWMODEL_URL = "http://127.0.0.1:8000/new_model/"
EVALUATE_URL = "http://127.0.0.1:8000/evaluate_model/"    

DUMMY_MODEL = "Dummy"
KNN_MODEL = "KNN"

metric = 0.0
key = 0

In [19]:
def send_get_request (url:str, params:dict) :
    """
    Send a GET request to the specified URL with the given parameters.
    Args:
        url (str): The URL to send the request to.
        params (dict): The parameters to include in the request.
    Returns:
        dict: The JSON response from the server.
    """
    data = {}
    response = requests.get(url, params=params)

    if response.status_code == 200:
        data = response.json()
        print(data)
    else:
        print(f"Error: {response.status_code}")
    return data

In [20]:
def get_prediction(key, x, y) :
    """
    Get the prediction from the server for the given key and input data.
    Args:
        key (int): The key to identify the model.
        x (list): The input data for prediction.
        y (list): The target data for prediction.
    Returns:
        str: The prediction result from the server.
    """
    params = {"id" : key, "x" :x, "y" :y}
    response = send_get_request(PREDICT_URL, params)
    return response.get("prediction")

In [21]:
def send_result(key, result) :
    """
    Send the result to the server for the given key.
    Args:
        key (int): The key to identify the model.
        result (str): The result to send to the server.
    Returns:
        dict: The JSON response from the server.
    """
    params = {"id" : key, "result" : result}
    response = send_get_request(RESULT_URL, params)
    return response

In [22]:
#Modified request retrain function to send proper request to server
def request_retrain(last_n: int, model_type="Dummy", strategy="stratified", k=3):
    return send_get_request(RETRAIN_URL, {
        "last_n": last_n,
        "model_type": model_type,
        "strategy": strategy,
        "k": k
    })

In [23]:
def request_new_model(model_name:str, model_params:dict, last_n:int) :
    """
    Request the server to create a new model with the given name, parameters, and number of last samples.
    Args:
        model_name (str): The name of the model to create.
        model_params (dict): The parameters for the model.
        last_n (int): The number of last samples to use for training.
    Returns:
        dict: The JSON response from the server.
    """
    params = {"last_n" : last_n, "model": model_name, "model_params":model_params}
    response = send_get_request(NEWMODEL_URL)
    return response

In [24]:
def request_evaluation(last_n:int) :
    """
    Request the server to evaluate the model with the given number of last samples.
    Args:
        last_n (int): The number of last samples to use for evaluation.
    Returns:
        dict: The JSON response from the server.
    """
    params = {"last_n" : last_n}
    response = send_get_request(EVALUATE_URL, params)
    return response

In [25]:
def new_dummy(strategy: str, last_n: int) :
    """
    Request the server to create a new dummy model with the given strategy and number of last samples.
    Args:
        strategy (str): The strategy for the dummy model.
        last_n (int): The number of last samples to use for training.
    Returns:
        dict: The JSON response from the server.
    """
    model_params = {"strategy" : strategy}
    response = request_new_model(DUMMY_MODEL, model_params, last_n) 
    return response

In [26]:
def new_knn(k:int, last_n: int):
    """
    Request the server to create a new KNN model with the given number of neighbors and last samples.
    Args:
        k (int): The number of neighbors for the KNN model.
        last_n (int): The number of last samples to use for training.
    Returns:
        dict: The JSON response from the server.
    """
    model_params = {"k" : k}
    response = request_new_model(KNN_MODEL, model_params, last_n)
    return response

In [27]:
def item(key,metric) :
    """
    Generate a new item with a unique key and random values for x, y, and z.
    Args:
        key (int): The unique key for the item.
        metric (float): The metric to compare against.
    Returns:
        list: A list containing the key, x, y, and result of the comparison.
    """
    key=key+1
    x=np.random.rand()*2-1
    y=np.random.rand()*2-1
    z=np.random.rand()*.2-.1
    result=x*y+z>metric
    return [key, x, y, result]

In [28]:
key, x, y, result = item(key, metric)

prediction = get_prediction(key, x, y)
send_result(key, result)

{'prediction': 'True'}
{'message': 'Copied result for 1'}


{'message': 'Copied result for 1'}

In [29]:
for _ in range(15):
    key, x, y, result = item(key, metric)
    prediction = get_prediction(key, x, y)
    send_result(key, result)
# request retrain with last 10 samples

{'prediction': 'True'}
{'message': 'Copied result for 2'}
{'prediction': 'True'}
{'message': 'Copied result for 3'}
{'prediction': 'True'}
{'message': 'Copied result for 4'}
{'prediction': 'True'}
{'message': 'Copied result for 5'}
{'prediction': 'True'}
{'message': 'Copied result for 6'}
{'prediction': 'True'}
{'message': 'Copied result for 7'}
{'prediction': 'True'}
{'message': 'Copied result for 8'}


{'prediction': 'True'}
{'message': 'Copied result for 9'}
{'prediction': 'True'}
{'message': 'Copied result for 10'}
{'prediction': 'True'}
{'message': 'Copied result for 11'}
{'prediction': 'True'}
{'message': 'Copied result for 12'}
{'prediction': 'True'}
{'message': 'Copied result for 13'}
{'prediction': 'True'}
{'message': 'Copied result for 14'}
{'prediction': 'True'}
{'message': 'Copied result for 15'}
{'prediction': 'True'}
{'message': 'Copied result for 16'}


In [30]:
request_retrain(last_n=15, model_type="KNN")

{'message': 'Model retrained using KNN', 'accuracy': 0.5, 'recall': 0.5, 'precision': 0.6666666666666666}


{'message': 'Model retrained using KNN',
 'accuracy': 0.5,
 'recall': 0.5,
 'precision': 0.6666666666666666}

In [31]:


# FastAPI endpoint definition (example)
# @app.post("/items/")
# async def create_item(item: Item):
#     return item

#url = "http://127.0.0.1:8000/items/"  # Replace with your actual URL
#data = {"name": "New Item", "description": "A new item description"}
#headers = {'Content-type': 'application/json'}
#response = requests.post(url, data=json.dumps(data), headers=headers)

#if response.status_code == 200:
#    data = response.json()
#    print(data)
#else:
#    print(f"Error: {response.status_code}")